In [0]:
%sql
WITH weekly AS (
   SELECT repo_id, date_trunc('week', event_date) AS week, SUM(stars) AS stars_wk
   FROM workspace.gold.fact_repo_daily_metrics
   GROUP BY repo_id, date_trunc('week', event_date) 
),
wow AS(
    SELECT repo_id, week, stars_wk,
    LAG(stars_wk) OVER (PARTITION BY repo_id ORDER BY week) AS stars_prev_wk
    FROM weekly
)
SELECT r.repo_name, w.stars_wk, w.stars_prev_wk,
w.stars_wk - COALESCE(w.stars_prev_wk, 0) AS star_delta
FROM wow w 
JOIN workspace.gold.dim_repo r ON r.repo_id = w.repo_id AND r.is_current = TRUE
WHERE w.stars_wk > 0
ORDER BY star_delta DESC LIMIT 20

In [0]:
%sql
SELECT event_date, language, active_repos, events, ROUND(share_of_events * 100, 2) AS pct_of_day FROM workspace.gold.fact_language_trends
ORDER BY event_date, events DESC

In [0]:
%sql
WITH actor_repo AS (
    SELECT DISTINCT f.repo_id, f.actor_id
    FROM workspace.gold.fact_events f 
    JOIN workspace.gold.dim_actor a ON a.actor_id = f.actor_id
    WHERE a.is_bot = FALSE
)
SELECT ar.repo_id,
SUM(CASE WHEN date_trunc('month', a.first_seen_date) = DATE'2026-06-01' THEN 1 ELSE 0 END) AS new_contributors,
SUM(CASE WHEN date_trunc('month', a.first_seen_date) < DATE'2026-06-01' THEN 1 ELSE 0 END) AS returning_contributors
FROM actor_repo ar
JOIN workspace.gold.dim_actor a ON a.actor_id = ar.actor_id
GROUP BY ar.repo_id
ORDER BY (new_contributors + returning_contributors) DESC LIMIT 20

In [0]:
%sql
SELECT d.day_name, F.event_hour, f.event_type, COUNT(*) AS events
FROM workspace.gold.fact_events f 
JOIN workspace.gold.dim_date d ON d.date_key = f.date_key
GROUP BY d.day_name, f.event_hour, f.event_type
ORDER BY events DESC LIMIT 20;

In [0]:
%sql
WITH recent AS (
    SELECT repo_id, SUM(stars) AS stars_7d
    FROM workspace.gold.fact_repo_daily_metrics
    WHERE event_date >= (SELECT MAX(event_date) FROM workspace.gold.fact_repo_daily_metrics) - INTERVAL 7 DAYS
    GROUP BY repo_id
),
total AS (
    SELECT repo_id, SUM(stars) AS stars_total
    FROM workspace.gold.fact_repo_daily_metrics GROUP BY repo_id
)
SELECT r.repo_name, rec.stars_7d, tot.stars_total,
    ROUND(rec.stars_7d / tot.stars_total, 3) AS growth_rate
FROM recent rec 
JOIN total tot ON tot.repo_id = rec.repo_id
JOIN workspace.gold.dim_repo r ON r.repo_id = rec.repo_id AND r.is_current = TRUE
WHERE tot.stars_total >= 20
ORDER BY rec.stars_7d DESC LIMIT 20


In [0]:
%sql
OPTIMIZE workspace.gold.fact_events ZORDER BY (repo_id);
OPTIMIZE workspace.gold.fact_repo_daily_metrics ZORDER BY (repo_id)